# Feature Engineering on Weather and Hourly Demand Datasets:

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_weather+demand")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/21 01:32:39 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.33.67 instead (on interface en0)
24/08/21 01:32:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/21 01:32:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/21 01:32:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Read Files:

In [3]:
base_dir = "../data"

Hourly weather dataset:

In [4]:
hourly_weather_sdf_path = base_dir + '/curated/weather_data/preprocessed_hourly_weather'
hourly_weather_sdf = spark.read.parquet(hourly_weather_sdf_path)
hourly_weather_sdf.show(5)

+----------+----+-------------+----------------------+-------------------+------------------+
|  DateOnly|Hour|AvgHourlyTemp|AvgHourlyPrecipitation|AvgHourlyVisibility|AvgHourlyWindSpeed|
+----------+----+-------------+----------------------+-------------------+------------------+
|2023-07-01|   0|         22.2|                   0.0|              9.656|               0.0|
|2023-07-01|   1|         21.7|                   0.0|              9.656|               2.6|
|2023-07-01|   2|         21.1|                   0.0|              9.656|               1.5|
|2023-07-01|   3|         21.1|                   0.0|             11.265|               1.5|
|2023-07-01|   4|         20.6|                   0.0|              9.656|               0.0|
+----------+----+-------------+----------------------+-------------------+------------------+
only showing top 5 rows



Hourly demand dataset:

In [5]:
hourly_demand_sdf_dir = base_dir + '/developed/merged_data/hourly_pickup_demand'
hourly_demand_sdf = spark.read.parquet(hourly_demand_sdf_dir)
hourly_demand_sdf.show(5)

+-----------+-----------+--------+------------------+
|pickup_hour|pickup_date|day_type|mean_hourly_demand|
+-----------+-----------+--------+------------------+
|          0| 2023-08-20| Weekend|1401.1666666666667|
|          0| 2023-08-19| Weekend|          1261.625|
|          0| 2023-08-27| Weekend|1409.8333333333333|
|          0| 2023-08-26| Weekend|1221.0416666666667|
|          0| 2023-08-25| Weekday|            698.25|
+-----------+-----------+--------+------------------+
only showing top 5 rows



# Aggregated Hourly Demand Dataset:

In [6]:
hourly_demand_sdf = hourly_demand_sdf.drop('day_type')
hourly_demand_sdf = hourly_demand_sdf.groupBy('pickup_hour').agg(
    F.avg('mean_hourly_demand').alias('avg_hourly_demand')
)
hourly_demand_sdf = hourly_demand_sdf.withColumnRenamed('pickup_hour', 'hour')

hourly_demand_sdf.show(5)

+----+------------------+
|hour| avg_hourly_demand|
+----+------------------+
|  12| 980.2212409420296|
|  22|1221.5695199275358|
|   1| 615.0425724637681|
|  13| 1016.955842391304|
|   6| 639.5403079710143|
+----+------------------+
only showing top 5 rows



# Merge Two Datasets:

In [7]:
# Rename the `Hour` column of `hourly_weather_sdf` to `hour` for merging
hourly_weather_sdf = hourly_weather_sdf.withColumnRenamed('Hour', 'hour')

# Merge by `hour`` column
hourly_demand_by_weather = hourly_demand_sdf.join(hourly_weather_sdf, on='hour', how='inner')

hourly_demand_by_weather.show(5)

+----+------------------+----------+-------------+----------------------+-------------------+------------------+
|hour| avg_hourly_demand|  DateOnly|AvgHourlyTemp|AvgHourlyPrecipitation|AvgHourlyVisibility|AvgHourlyWindSpeed|
+----+------------------+----------+-------------+----------------------+-------------------+------------------+
|   0| 870.5163043478262|2023-07-01|         22.2|                   0.0|              9.656|               0.0|
|   1| 615.0425724637681|2023-07-01|         21.7|                   0.0|              9.656|               2.6|
|   2|444.17912137681157|2023-07-01|         21.1|                   0.0|              9.656|               1.5|
|   3| 355.9535778985509|2023-07-01|         21.1|                   0.0|             11.265|               1.5|
|   4|371.67232789855046|2023-07-01|         20.6|                   0.0|              9.656|               0.0|
+----+------------------+----------+-------------+----------------------+-------------------+---

# Save the Merged Dataset:

In [8]:
hourly_demand_by_weather_dir = base_dir + '/developed/merged_data'
file_name = 'hourly_demand_by_weather'
hourly_demand_by_weather_path = os.path.join(hourly_demand_by_weather_dir, file_name)
hourly_demand_by_weather.write.mode('overwrite').parquet(hourly_demand_by_weather_path)